# 07 (v2.1). 재정렬 파인튜닝 + 걸러진 데이터로 LoRA 재학습 (06 개선판)

06 문장형 LoRA는 ROUGE-L(문장형)이 0.33 → 0.56으로 올랐지만, **정답 포함률은 0.53 → 0.47로 떨어졌습니다.**
원인: 한 문장에 문서 하나의 사실만 담고, 정답 문서를 인용한 비율이 39%(28/71)뿐이었습니다.
이 노트북은 두 가지를 한 번에 고칩니다.

| 개선 | 내용 |
|---|---|
| ① 재정렬 파인튜닝 | MiniLM-L-6을 v2.1 train 건강 질문으로 추가 학습 (정답 문단 1 + bge가 헷갈린 오답 7). 검증에서 좋아질 때만 사용 |
| ② 학습 데이터 거르기·늘리기 | 짧은 정답 내용이 빠진 엉터리 문장형 정답은 제외, 쓸 수 있는 데이터를 최대한 사용 |

```
① 재정렬 학습 ─▶ ② 새 재정렬로 검색한 문단 5개 + "걸러진 문장형 정답 [번호]" ─▶ LoRA 학습 ─▶ 평가
```

**평가 (각 개선의 효과를 나눠서 봄)**

| 시스템 | 재정렬 | 생성 모델 |
|---|---|---|
| 원본 Qwen | 기존 | LoRA 없음 |
| 06 문장형 LoRA | 기존 | 06 |
| 06 문장형 LoRA + 새 재정렬 | **새것** | 06 → ①의 효과 |
| **07 (전부)** | **새것** | **07** → ②까지의 효과 |

**런타임: A100 + High-RAM.** 위에서부터 Run all. 전체 약 1시간 (DB 불러오기 5~8분, 재정렬 학습 5~10분, LoRA 학습 20~25분, 평가 15분).
결과는 드라이브 `rag_msmarco/health/upgrade07/`와 `RAG/2_results/upgrade07/`에 저장되고, 마지막 셀이 런타임을 끕니다.

## 1. 설치 + 저장 위치

In [ ]:
!pip -q install -U sentence-transformers faiss-cpu peft datasets
!pip -q uninstall -y torchao    # Colab 기본 torchao(0.10)가 최신 peft와 충돌해서 제거

In [ ]:
import os
from google.colab import drive
drive.mount('/content/drive')
SAVE_DIR    = '/content/drive/MyDrive/rag_msmarco/health'          # v2.1 DB (02에서 만든 것)
OUT_DIR     = f'{SAVE_DIR}/upgrade07'
REPORT_DIR  = '/content/drive/MyDrive/RAG/2_results/upgrade07'      # 결과 요약 사본 (Mac RAG 폴더와 동기화)
LORA_06     = f'{SAVE_DIR}/lora_sentence/adapter'                   # 06 문장형 LoRA
os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(REPORT_DIR, exist_ok=True)
print('결과 저장:', OUT_DIR, '/ 06 LoRA 있음:', os.path.isdir(LORA_06))

## 2. 설정

시간이 밀리면 `N_TRAIN`(LoRA)과 `RR_N_Q`(재정렬)부터 줄이세요.

In [ ]:
import re, json, gzip, time, random, string
from collections import Counter
from urllib.parse import urlparse
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F

SEED       = 42
K          = 100        # 1단계 후보 수 (= 재정렬할 후보 수)
TOP_N      = 5          # LLM에 넣을 문단 수
GEN_BATCH  = 16
MAX_NEW    = 256

# ① 재정렬 파인튜닝
RR_N_Q     = 8000       # 재정렬 학습 질문 수
RR_NEG     = 7          # 질문당 오답 문단 수 (bge 상위 후보 중 정답이 아닌 것)
RR_BATCH   = 16         # 질문 묶음 (한 번에 16 x 8 = 128쌍)
RR_LR      = 1e-5
RR_EPOCHS  = 1
RR_MIN_GAIN = 0.0       # 검증 MRR이 이만큼 이상 좋아져야 새 재정렬을 사용

# ② ③ LoRA
N_TRAIN       = 4000    # 학습 예시 최대 수 (거절 예시 포함)
REFUSE_FRAC   = 0.15
MIN_WF_RECALL = 0.5     # 문장형 정답에 짧은 정답 단어가 이 비율 이상 들어 있어야 사용 (엉터리 정답 제외)
ADD_SECOND    = 0.0     # 두 번째 문장 붙이기 끔 (확인 결과 대부분 첫 문장의 다른 표현이라 반복만 배움). 켜려면 0.8
MAX_LEN       = 1024
EPOCHS        = 1
LR            = 2e-4
BATCH, GRAD_ACC = 8, 2
LORA_R, LORA_ALPHA = 16, 32

EMB_MODEL  = 'BAAI/bge-base-en-v1.5'
QUERY_PREFIX = 'Represent this sentence for searching relevant passages: '
RERANKER   = 'cross-encoder/ms-marco-MiniLM-L-6-v2'
LLM_NAME   = 'Qwen/Qwen2.5-7B-Instruct'
NO_ANSWER  = 'No Answer Present.'
REFUSAL    = "Sorry, I can't answer that from the medical documents I have."

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
assert DEVICE == 'cuda', 'GPU 런타임이 아닙니다. 런타임 유형을 A100으로 바꾸세요.'
print('GPU:', torch.cuda.get_device_name(0), f'{torch.cuda.get_device_properties(0).total_memory / 1e9:.0f}GB')

def load_jsonl(path):
    opener = gzip.open if path.endswith('.gz') else open
    with opener(path, 'rt', encoding='utf-8') as f:
        return [json.loads(line) for line in f]

def save_jsonl(rows, path):
    with open(path, 'w', encoding='utf-8') as f:
        for r in rows:
            f.write(json.dumps(r, ensure_ascii=False) + '\n')

def toks(s):
    s = s.lower()
    s = re.sub(r'\[\d+\]', ' ', s)
    s = ''.join(ch if ch not in string.punctuation else ' ' for ch in s)
    return [w for w in s.split() if w not in {'a', 'an', 'the'}]

def recall(pred, ref):
    # 짧은 정답의 단어 중 pred에 들어 있는 비율
    p, r = Counter(toks(pred)), Counter(toks(ref))
    return sum((p & r).values()) / max(1, sum(r.values()))

## 3. DB와 검색기

재정렬 모델은 sentence-transformers 버전 차이를 피하려고 transformers로 직접 불러 점수(logit)를 계산합니다. 04~06의 점수와 같은 값입니다.

In [ ]:
import faiss
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForSequenceClassification

t = time.time()
corpus   = load_jsonl(f'{SAVE_DIR}/corpus.jsonl')
eval_set = load_jsonl(f'{SAVE_DIR}/eval.jsonl')
val_set  = load_jsonl(f'{SAVE_DIR}/rerank/val.jsonl')
index    = faiss.read_index(f'{SAVE_DIR}/faiss_bge_flatip.index')
texts    = [c['text'] for c in corpus]
assert index.ntotal == len(corpus), '인덱스와 문단 수가 다릅니다'
print(f'문단 {len(corpus):,} / 평가 {len(eval_set)} / 검증 {len(val_set)} ({time.time() - t:.0f}초)')

emb_model = SentenceTransformer(EMB_MODEL, device=DEVICE)
emb_model.half()
rr_tok = AutoTokenizer.from_pretrained(RERANKER)

def load_reranker(path):
    m = AutoModelForSequenceClassification.from_pretrained(path).to(DEVICE).eval()
    return m.half()

rr_base = load_reranker(RERANKER)

def site_of(url):
    h = urlparse(url).netloc.lower()
    return h[4:] if h.startswith('www.') else h

sites = [site_of(c['url']) for c in corpus]
norm_text = lambda t: ' '.join(t.lower().split())

def encode(strings):
    return emb_model.encode([QUERY_PREFIX + s for s in strings], batch_size=128, normalize_embeddings=True,
                            convert_to_numpy=True).astype('float32')

def dense_search(queries, k=K):
    return index.search(encode(queries), k)[1].tolist()

@torch.inference_mode()
def rr_score(model, pairs, bs=512):
    out = []
    for i in range(0, len(pairs), bs):
        q, p = zip(*pairs[i:i + bs])
        enc = rr_tok(list(q), list(p), truncation=True, max_length=256, padding=True, return_tensors='pt').to(DEVICE)
        out.append(model(**enc).logits.float().view(-1).cpu())
    return torch.cat(out).numpy()

def retrieve_many(queries, model, top_n=TOP_N, cands=None):
    # 질문마다 [{pid, score, top_raw, site, text}, ...] 상위 top_n개 (본문이 같은 문단은 하나만)
    cands = cands or dense_search(queries)
    pairs = [(q, texts[p]) for q, cs in zip(queries, cands) for p in cs]
    all_scores = rr_score(model, pairs).reshape(len(queries), K)
    out = []
    for cs, scores in zip(cands, all_scores):
        docs, seen = [], set()
        for i in np.argsort(-scores):
            p = int(cs[i])
            key = norm_text(texts[p])
            if key in seen:
                continue
            seen.add(key)
            docs.append({'pid': p, 'score': float(scores[i]), 'top_raw': float(scores.max()),
                         'site': sites[p], 'text': texts[p]})
            if len(docs) == top_n:
                break
        out.append(docs)
    return out

def rank_metrics(qs, docs_list):
    r1 = np.mean([d[0]['pid'] in set(q['gold_pids']) for q, d in zip(qs, docs_list)])
    r5 = np.mean([bool(set(q['gold_pids']) & {x['pid'] for x in d}) for q, d in zip(qs, docs_list)])
    mrr = np.mean([next((1 / i for i, x in enumerate(d, 1) if x['pid'] in set(q['gold_pids'])), 0) for q, d in zip(qs, docs_list)])
    return {'R@1': round(r1, 3), 'R@5': round(r5, 3), 'MRR@5': round(mrr, 3)}

# 검증·시험 질문은 bge 후보를 한 번만 계산해 두고 재정렬 모델만 바꿔 비교
val_q   = [q['query'] for q in val_set]
test_rr = [q for q in eval_set if q['domain'] == 'health' and q['answerable']]      # 답 있는 건강 500
val_c, test_c = dense_search(val_q), dense_search([q['query'] for q in test_rr])
base_val  = rank_metrics(val_set, retrieve_many(val_q, rr_base, cands=val_c))
base_test = rank_metrics(test_rr, retrieve_many([q['query'] for q in test_rr], rr_base, cands=test_c))
print('기존 재정렬  검증:', base_val, '/ 시험:', base_test)

## 4. ① 재정렬 파인튜닝

- train 건강 질문(문단이 DB에 있는 것)마다 **정답 문단 1개 + bge 상위 후보 중 정답이 아닌 문단 7개**를 묶어, 정답이 1등이 되도록 학습합니다 (listwise softmax).
- MS MARCO는 정답 표시가 하나뿐이라, 오답 후보 중 **짧은 정답이 거의 그대로 들어 있는 문단은 오답에서 뺍니다** (사실은 정답일 수 있으므로).
- 검증 MRR이 좋아졌을 때만 새 재정렬을 씁니다. 아니면 기존 재정렬을 그대로 씁니다.

In [ ]:
pid_of = {t: i for i, t in enumerate(texts)}
held_out = {q['qid'] for q in eval_set} | {q['qid'] for q in val_set}

train_qs = []
with gzip.open(f'{SAVE_DIR}/train_health_qa.jsonl.gz', 'rt', encoding='utf-8') as f:
    for line in f:
        q = json.loads(line)
        if not q['in_corpus'] or q['qid'] in held_out:
            continue
        gold = {pid_of[p['text']] for p in q['passages'] if p['is_selected'] == 1 and p['text'] in pid_of}
        train_qs.append({'qid': q['qid'], 'query': q['query'], 'answers': q['answers'],
                         'well_formed': q['well_formed'], 'gold': gold})
print(f'train 건강 질문 (문단이 DB에 있음): {len(train_qs):,}개')

rng = random.Random(SEED)
rr_pool = [q for q in train_qs if q['gold'] and q['answers'] != [NO_ANSWER]]
rng.shuffle(rr_pool)
rr_pool = rr_pool[:RR_N_Q]

t = time.time()
rr_cands = dense_search([q['query'] for q in rr_pool], k=30)
groups = []
for q, cs in zip(rr_pool, rr_cands):
    ans = q['answers'][0]
    negs = [p for p in cs if p not in q['gold'] and not (len(toks(ans)) <= 30 and recall(texts[p], ans) >= 0.8)]
    if len(negs) < RR_NEG:
        continue
    pos = rng.choice(sorted(q['gold']))
    groups.append((q['query'], [texts[pos]] + [texts[p] for p in rng.sample(negs[:20], RR_NEG)]))
print(f'재정렬 학습 묶음 {len(groups):,}개 ({time.time() - t:.0f}초)')

In [ ]:
from transformers import get_linear_schedule_with_warmup

rr_ft = AutoModelForSequenceClassification.from_pretrained(RERANKER).to(DEVICE).train()
opt = torch.optim.AdamW(rr_ft.parameters(), lr=RR_LR, weight_decay=0.01)
n_steps = -(-len(groups) // RR_BATCH) * RR_EPOCHS
sched = get_linear_schedule_with_warmup(opt, int(n_steps * 0.1), n_steps)

t, step, losses = time.time(), 0, []
for ep in range(RR_EPOCHS):
    random.Random(SEED + ep).shuffle(groups)
    for i in range(0, len(groups), RR_BATCH):
        b = groups[i:i + RR_BATCH]
        enc = rr_tok([q for q, ps in b for _ in ps], [p for _, ps in b for p in ps],
                     truncation=True, max_length=256, padding=True, return_tensors='pt').to(DEVICE)
        with torch.autocast('cuda', dtype=torch.bfloat16):
            logits = rr_ft(**enc).logits.view(len(b), -1).float()
        loss = F.cross_entropy(logits, torch.zeros(len(b), dtype=torch.long, device=DEVICE))
        loss.backward()
        torch.nn.utils.clip_grad_norm_(rr_ft.parameters(), 1.0)
        opt.step(); sched.step(); opt.zero_grad()
        losses.append(loss.item()); step += 1
        if step % 50 == 0:
            print(f'  {step}/{n_steps}  loss {np.mean(losses[-50:]):.3f}')
print(f'재정렬 학습 끝: {(time.time() - t) / 60:.1f}분')

RR_DIR = f'{OUT_DIR}/reranker'
rr_ft.save_pretrained(RR_DIR); rr_tok.save_pretrained(RR_DIR)
rr_ft = rr_ft.eval().half()

ft_val  = rank_metrics(val_set, retrieve_many(val_q, rr_ft, cands=val_c))
ft_test = rank_metrics(test_rr, retrieve_many([q['query'] for q in test_rr], rr_ft, cands=test_c))
rr_table = pd.DataFrame({'기존 (검증)': base_val, '새것 (검증)': ft_val, '기존 (시험 500)': base_test, '새것 (시험 500)': ft_test}).T
display(rr_table)

USE_FT = ft_val['MRR@5'] - base_val['MRR@5'] > RR_MIN_GAIN
rr_new = rr_ft if USE_FT else rr_base
print('새 재정렬 사용:', USE_FT, '(검증 MRR 기준으로 결정, 시험 점수는 보고만 함)')

def thresh_of(model):
    return float(np.percentile([d[0]['top_raw'] for d in retrieve_many(val_q, model, cands=val_c)], 5))
TH_BASE, TH_NEW = thresh_of(rr_base), thresh_of(rr_new)
print(f'거절 기준 (검증 하위 5%): 기존 {TH_BASE:.2f} / 새것 {TH_NEW:.2f}')

## 5. ② LoRA 학습 데이터 (걸러진 문장형 정답)

- **거르기**: 문장형 정답에 짧은 정답 단어가 `MIN_WF_RECALL`(50%) 미만이면 제외
- **두 번째 문장 (기본 꺼짐)**: `ADD_SECOND`를 0.8로 바꾸면, 문장형 정답에 짧은 정답 내용이 덜 담긴 경우 짧은 정답 문장을 붙임. 데이터를 확인해 보니 대부분 같은 내용의 반복이라 끔
- **출처**: 정답 문단이 상위 5개 중 몇 번째인지로 [번호]. 정답 문단이 상위 5개 밖이면 학습에서 제외
- **거절 예시**: `No Answer Present.` 질문 → 거절 문장
- 검색은 위에서 고른 재정렬(`rr_new`)로 합니다 (실제 사용 때와 같게).

In [ ]:
def is_sentence(s):
    return len(s.split()) >= 6 and s.rstrip().endswith('.')

def cite(sentence, nums):
    tag = ''.join(f'[{n}]' for n in nums)
    s = sentence.strip()
    return s[:-1] + f' {tag}.' if s.endswith('.') else f'{s} {tag}.'

wf_pool, noans_pool, n_bad, n_two = [], [], 0, 0
for q in train_qs:
    if q['answers'] == [NO_ANSWER]:
        noans_pool.append(q)
        continue
    if not (q['well_formed'] and q['gold']):
        continue
    wf, short = q['well_formed'][0], q['answers'][0]
    cov = recall(wf, short)
    if cov < MIN_WF_RECALL:
        n_bad += 1
        continue
    second = short if (cov < ADD_SECOND and is_sentence(short) and norm_text(short) != norm_text(wf)) else None
    n_two += second is not None
    wf_pool.append({**q, 'first': wf, 'second': second})
print(f'문장형 정답 사용 {len(wf_pool):,}개 (엉터리로 제외 {n_bad:,}, 두 문장 {n_two:,}) / 답 없음 {len(noans_pool):,}개')

rng = random.Random(SEED)
rng.shuffle(wf_pool); rng.shuffle(noans_pool)
n_ref = int(N_TRAIN * REFUSE_FRAC)
noans_pick = noans_pool[:n_ref]

t = time.time()
wf_docs    = retrieve_many([q['query'] for q in wf_pool], rr_new)
noans_docs = retrieve_many([q['query'] for q in noans_pick], rr_new)
print(f'검색 {len(wf_pool) + len(noans_pick):,}개: {time.time() - t:.0f}초')

answerable = []
for q, docs in zip(wf_pool, wf_docs):
    nums = [i for i, d in enumerate(docs, 1) if d['pid'] in q['gold']]
    if not nums:
        continue
    target = cite(q['first'], nums) + (' ' + cite(q['second'], nums) if q['second'] else '')
    answerable.append({'qid': q['qid'], 'query': q['query'], 'pids': [d['pid'] for d in docs], 'target': target})
refusals = [{'qid': q['qid'], 'query': q['query'], 'pids': [d['pid'] for d in docs], 'target': REFUSAL}
            for q, docs in zip(noans_pick, noans_docs)]

train_ex = answerable[:N_TRAIN - len(refusals)] + refusals
random.Random(SEED).shuffle(train_ex)
save_jsonl(train_ex, f'{OUT_DIR}/train_examples.jsonl')
n_ans = len(train_ex) - len(refusals)
print(f'학습 예시 {len(train_ex):,}개 (답변 {n_ans:,} + 거절 {len(refusals):,})')
for ex in [e for e in train_ex if e['target'].count('[') >= 2][:3]:
    print(' -', ex['query'], '→', ex['target'][:220])

## 6. LoRA 학습

05·06과 같은 설정과 프롬프트입니다. 체크포인트는 드라이브 `upgrade07/ckpt/`에 저장됩니다.

In [ ]:
from transformers import AutoModelForCausalLM, Trainer, TrainingArguments
from peft import LoraConfig, get_peft_model

SYS_RAG = f'''You are a careful medical information assistant.
Rules:
1. Answer ONLY with facts stated in the numbered documents. Do not add outside knowledge.
2. After each sentence, cite the supporting document numbers like [1] or [2][4].
3. If the question is not about health or medicine, or the documents do not contain the answer, reply with exactly this sentence: "{REFUSAL}"
4. Answer in 2-4 short sentences.'''

def rag_messages(q, docs):
    ctx = '\n'.join(f'[{i}] ({d["site"]}) {d["text"]}' for i, d in enumerate(docs, 1))
    return [{'role': 'system', 'content': SYS_RAG},
            {'role': 'user', 'content': f'Documents:\n{ctx}\n\nQuestion: {q}'}]

tok = AutoTokenizer.from_pretrained(LLM_NAME)
llm = AutoModelForCausalLM.from_pretrained(LLM_NAME, dtype=torch.bfloat16, device_map='cuda')
llm.config.use_cache = False
llm.gradient_checkpointing_enable(gradient_checkpointing_kwargs={'use_reentrant': False})
llm.enable_input_require_grads()
llm = get_peft_model(llm, LoraConfig(r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=0.05, task_type='CAUSAL_LM',
                                     target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj',
                                                     'gate_proj', 'up_proj', 'down_proj']))
llm.print_trainable_parameters()

def to_features(ex):
    docs = [{'site': sites[p], 'text': texts[p]} for p in ex['pids']]
    prompt = tok.apply_chat_template(rag_messages(ex['query'], docs), tokenize=False, add_generation_prompt=True)
    p_ids = tok(prompt, add_special_tokens=False)['input_ids']
    a_ids = tok(ex['target'] + '<|im_end|>', add_special_tokens=False)['input_ids']
    return {'input_ids': p_ids + a_ids, 'labels': [-100] * len(p_ids) + a_ids}

feats = [f for f in map(to_features, train_ex) if len(f['input_ids']) <= MAX_LEN]
print(f'학습 예시 {len(feats):,}개 / 평균 길이 {np.mean([len(f["input_ids"]) for f in feats]):.0f} 토큰')

def collate(batch):
    n = max(len(f['input_ids']) for f in batch)
    pad = lambda xs, v: [x + [v] * (n - len(x)) for x in xs]
    ids = pad([f['input_ids'] for f in batch], tok.pad_token_id)
    return {'input_ids': torch.tensor(ids),
            'attention_mask': torch.tensor(pad([[1] * len(f['input_ids']) for f in batch], 0)),
            'labels': torch.tensor(pad([f['labels'] for f in batch], -100))}

In [ ]:
CKPT_DIR = f'{OUT_DIR}/ckpt'
total_steps = -(-len(feats) // (BATCH * GRAD_ACC)) * EPOCHS
args = TrainingArguments(            # transformers 5: warmup_ratio 없음 → warmup_steps로 3%
    output_dir=CKPT_DIR, num_train_epochs=EPOCHS, learning_rate=LR, lr_scheduler_type='cosine',
    warmup_steps=max(1, int(total_steps * 0.03)),
    per_device_train_batch_size=BATCH, gradient_accumulation_steps=GRAD_ACC, bf16=True,
    logging_steps=10, save_steps=50, save_total_limit=2, report_to='none',
    remove_unused_columns=False, seed=SEED)
trainer = Trainer(model=llm, args=args, train_dataset=feats, data_collator=collate)

has_ckpt = os.path.isdir(CKPT_DIR) and any(d.startswith('checkpoint-') for d in os.listdir(CKPT_DIR))
t = time.time()
trainer.train(resume_from_checkpoint=has_ckpt or None)
print(f'학습 끝: {(time.time() - t) / 60:.0f}분')

ADAPTER_DIR = f'{OUT_DIR}/adapter'
llm.save_pretrained(ADAPTER_DIR)
tok.save_pretrained(ADAPTER_DIR)
pd.DataFrame([l for l in trainer.state.log_history if 'loss' in l]).to_csv(f'{OUT_DIR}/train_log.csv', index=False)
print('LoRA 저장:', ADAPTER_DIR)

## 7. 평가

06과 같은 질문입니다: 문장형 정답이 있는 건강 질문 + 답 없는 건강 질문 100 + 분야 밖 50.
재정렬이 바뀌면 거절 기준도 그 재정렬의 검증 점수로 다시 정합니다.

In [ ]:
import string

def toks(s):
    s = s.lower()
    s = re.sub(r'\[\d+\]', ' ', s)                          # 출처 번호는 채점에서 제외
    s = ''.join(ch if ch not in string.punctuation else ' ' for ch in s)
    return [w for w in s.split() if w not in {'a', 'an', 'the'}]

def f1(pred, ref):
    p, r = toks(pred), toks(ref)
    common = sum((Counter(p) & Counter(r)).values())
    if not p or not r or common == 0:
        return 0.0
    prec, rec = common / len(p), common / len(r)
    return 2 * prec * rec / (prec + rec)

def rouge_l(pred, ref):
    p, r = toks(pred), toks(ref)
    if not p or not r:
        return 0.0
    dp = [0] * (len(r) + 1)
    for a in p:
        prev = 0
        for j, b in enumerate(r, 1):
            cur = dp[j]
            dp[j] = prev + 1 if a == b else max(dp[j], dp[j - 1])
            prev = cur
    lcs = dp[-1]
    if lcs == 0:
        return 0.0
    prec, rec = lcs / len(p), lcs / len(r)
    return 2 * prec * rec / (prec + rec)

best = lambda fn, pred, refs: max(fn(pred, r) for r in refs)
has_cite = lambda a: bool(re.search(r'\[\d\]', a))

In [ ]:
def is_refusal(ans):
    a = ans.lower()
    return any(p in a for p in ["can't answer", 'cannot answer', 'unable to answer'])

llm.eval()
llm.config.use_cache = True
tok.padding_side = 'left'

@torch.inference_mode()
def generate(msgs_list, bs=GEN_BATCH):
    outs = []
    for i in range(0, len(msgs_list), bs):
        prompts = [tok.apply_chat_template(m, tokenize=False, add_generation_prompt=True) for m in msgs_list[i:i + bs]]
        enc = tok(prompts, return_tensors='pt', padding=True).to(llm.device)
        gen = llm.generate(**enc, max_new_tokens=MAX_NEW, do_sample=False, temperature=None, top_p=None, top_k=None,
                           pad_token_id=tok.pad_token_id)
        outs += tok.batch_decode(gen[:, enc['input_ids'].shape[1]:], skip_special_tokens=True)
    return [o.strip() for o in outs]

test_a = [q for q in eval_set if q['domain'] == 'health' and q['answerable'] and q['well_formed']]
test_n = [q for q in eval_set if q['domain'] == 'health' and not q['answerable']]
test_o = [q for q in eval_set if q['domain'] != 'health']
qs = test_a + test_n + test_o
queries = [q['query'] for q in qs]
cands = dense_search(queries)
docs_by_rr = {'기존': retrieve_many(queries, rr_base, cands=cands), '새것': retrieve_many(queries, rr_new, cands=cands)}
th_by_rr = {'기존': TH_BASE, '새것': TH_NEW}
print(f'평가: 문장형 정답 있는 건강 {len(test_a)} + 답 없음 {len(test_n)} + 분야 밖 {len(test_o)}')

has_06 = os.path.isdir(LORA_06)
if has_06:
    llm.load_adapter(LORA_06, adapter_name='s06')

systems = [('원본 Qwen', '기존', None)]
if has_06:
    systems += [('06 문장형 LoRA', '기존', 's06'), ('06 + 새 재정렬', '새것', 's06')]
systems += [('07 (새 재정렬 + 07 LoRA)', '새것', 'default')]

runs = {}
for name, rr, adapter in systems:
    t = time.time()
    docs_all = docs_by_rr[rr]
    todo = [i for i, d in enumerate(docs_all) if d[0]['top_raw'] >= th_by_rr[rr]]
    msgs = [rag_messages(queries[i], docs_all[i]) for i in todo]
    if adapter is None:
        with llm.disable_adapter():
            gen = generate(msgs)
    else:
        llm.set_adapter(adapter)
        gen = generate(msgs)
    answers = [REFUSAL] * len(qs)
    for i, a in zip(todo, gen):
        answers[i] = a
    runs[name] = {'answers': answers, 'docs': docs_all, 'sec': (time.time() - t) / len(qs)}
    print(f"{name} 끝 ({runs[name]['sec']:.2f}초/질문)")
llm.set_adapter('default')

In [ ]:
na, nn = len(test_a), len(test_n)
A_idx, N_idx, O_idx = range(na), range(na, na + nn), range(na + nn, len(qs))
gold_cite = lambda a, docs, q: bool({int(n) for n in re.findall(r'\[(\d)\]', a)} &
                                    {i for i, d in enumerate(docs, 1) if d['pid'] in set(q['gold_pids'])})
rows = {}
for name, r in runs.items():
    A, D = r['answers'], r['docs']
    answered = [i for i in A_idx if not is_refusal(A[i])]
    rows[name] = {
        'ROUGE-L (문장형)': np.mean([best(rouge_l, A[i], qs[i]['well_formed']) for i in A_idx]),
        '정답 포함률':     np.mean([best(recall, A[i], qs[i]['answers']) for i in A_idx]),
        'F1 (짧은 정답)':  np.mean([best(f1, A[i], qs[i]['answers']) for i in A_idx]),
        '근거 R@5':        np.mean([bool(set(qs[i]['gold_pids']) & {d['pid'] for d in D[i]}) for i in A_idx]),
        '정답 문서 인용':   np.mean([gold_cite(A[i], D[i], qs[i]) for i in A_idx]),
        '출처 표시율':     np.mean([has_cite(A[i]) for i in answered]) if answered else np.nan,
        '잘못 거절':       np.mean([is_refusal(A[i]) for i in A_idx]),
        '답 없음 거절률':  np.mean([is_refusal(A[i]) for i in N_idx]),
        '분야 밖 거절률':  np.mean([is_refusal(A[i]) for i in O_idx]),
        '평균 답변 단어':  np.mean([len(A[i].split()) for i in answered]) if answered else np.nan,
        '초/질문':        r['sec'],
    }
table = pd.DataFrame(rows).T.round(3)
display(table)

summary = '\n'.join([
    f"# 07 재정렬 파인튜닝 + 걸러진 데이터 LoRA (v2.1) ({time.strftime('%Y-%m-%d %H:%M', time.localtime(time.time() + 9 * 3600))} KST)", '',
    f'- ① 재정렬: MiniLM-L-6을 train 건강 질문 {len(groups):,}개(정답 1 + 오답 {RR_NEG})로 {RR_EPOCHS} epoch 학습 → 사용 여부 {USE_FT} (검증 MRR 기준)',
    f'- ② LoRA: 학습 {len(feats):,}개 (답변 {n_ans:,} + 거절 {len(refusals):,}), 엉터리 문장형 정답 제외 {n_bad:,}, 두 문장 정답 {n_two:,}',
    f'- 거절 기준 (검증 하위 5%): 기존 재정렬 {TH_BASE:.2f} / 새 재정렬 {TH_NEW:.2f}',
    f'- 평가: 문장형 정답 있는 건강 {na} + 답 없음 {nn} + 분야 밖 {len(test_o)}', '',
    '## ① 재정렬 비교 (상위 5개 기준)', '', rr_table.to_markdown(), '',
    '## 답변 비교', '', table.to_markdown(),
])
for d in (OUT_DIR, REPORT_DIR):
    with open(f'{d}/results_summary.md', 'w', encoding='utf-8') as f:
        f.write(summary)
    save_jsonl([{'qid': q['qid'], 'domain': q['domain'], 'answerable': q['answerable'], 'query': q['query'],
                 'gold_answer': (q.get('answers') or [''])[0], 'well_formed': (q.get('well_formed') or [''])[0],
                 **{name: r['answers'][i] for name, r in runs.items()},
                 **{f'{name} 문서': [d['pid'] for d in r['docs'][i]] for name, r in runs.items()}}
                for i, q in enumerate(qs)], f'{d}/eval_answers.jsonl')
pd.read_csv(f'{OUT_DIR}/train_log.csv').to_csv(f'{REPORT_DIR}/train_log.csv', index=False)
print('저장:', OUT_DIR, '/', REPORT_DIR)

In [ ]:
# 답변 비교 예시 (번호를 바꿔 보세요. 앞쪽은 답 있는 건강 질문)
i = 0
print('질문:', qs[i]['query'])
print('문장형 정답:', (qs[i].get('well_formed') or ['-'])[0][:200])
print('짧은 정답 :', (qs[i].get('answers') or ['-'])[0][:200])
for name, r in runs.items():
    print(f'\n[{name}]', r['answers'][i])

## 8. 드라이브 저장 마무리 + 런타임 종료

새 재정렬은 `upgrade07/reranker/`, LoRA는 `upgrade07/adapter/`에 있습니다.

In [ ]:
from google.colab import drive, runtime
drive.flush_and_unmount()
runtime.unassign()